# Quantum GAN — Detailed Notes (Session 21)
**Course:** CS490/5590 — Quantum Computing Applications in Data Science, AI, & Deep Learning  
**Instructor:** Luke Miller

> **Purpose.** Turn the slide bullets into a stand-alone reference on **quantum GANs (QGANs)**. We cover design choices (quantum generator/discriminator), building training loops with Qiskit primitives + PyTorch, losses (BCE/Wasserstein), barren-plateau mitigations, small datasets (1D mixtures, Bars-and-Stripes), evaluation metrics, and pitfalls on NISQ devices.

---

## Session roadmap
1. Recap: from quantum K-means to generative models  
2. What is a QGAN? (design choices & data models)  
3. Quantum **generators**: PQCs that sample bitstrings or amplitudes  
4. Quantum **discriminators**: PQCs that score real vs fake  
5. Variational training: objectives, gradients, optimizers  
6. Stabilization: mode collapse & barren plateaus  
7. Architectures for small datasets (1D GM, BAS(2×2/3×3))  
8. Qiskit implementations (modern primitives + TorchConnector)  
9. Assessing samples & reporting results  
10. Practical tips, pitfalls, mini-exercises

---

## 0) Recap → Why generative models after clustering?
- K-means **assigns**; GANs **synthesize**.  
- Quantum circuits are naturally **stochastic** (measurements) and live in exponentially large Hilbert spaces → compact parametrizations of complex distributions.  
- Synthetic data can augment scarce domains and create “pre-clustered” inputs.

---

## 1) What counts as a “Quantum GAN”?
You choose which parts are quantum:

| Variant | Generator | Discriminator | When to use |
|---|---|---|---|
| **QG-CD** | **Quantum** (PQC) | **Classical** (MLP/CNN) | Produce classical samples; leverage robust classical discriminator |
| **CG-QD** | Classical | **Quantum** (PQC) | When quantum kernels help separate modes |
| **QG-QD** | **Quantum** | **Quantum** | Research/teaching demos; end-to-end quantum

**Data models.**  
- **Classical distribution over bitstrings** $p(x)$, $x\in\{0,1\}^n$: generator prepares $|\psi(\theta)\rangle$; measurement yields samples $\tilde{x}$.  
- **Quantum state distribution** (quantum data): match a target density matrix/state fidelity (advanced).

Most labs start with **classical bitstring targets**.

---

## 2) Quantum generator $G(\theta)$
- **Input:** noise (optional) + parameter vector $\theta$. Noise can be classical (re-upload angles) or quantum (extra qubits).  
- **Circuit:** shallow **TwoLocal** (e.g., `ry/rz` + `cx` or `cz`) with 1–2 reps; data re-upload if you include noise $z$.  
- **Output:** measurement distribution over $\{0,1\}^n$.  
- **Expressivity:** add entanglers matching hardware topology; keep depth NISQ-safe.

**Bitstring likelihood.** For logit/prob estimates, use either:
- **Sampler**: get bitstring frequencies; or
- **Estimator**: treat specific outcomes with projectors (e.g., $|x\rangle\langle x|$) as observables (costlier to compose).

---

## 3) Quantum discriminator $D(\phi)$
- **Role:** score input as “real” (1) vs “fake” (0).  
- **Input prep:**  
  - **Real**: prepare $|x\rangle$ via `X` on bits set to 1.  
  - **Fake**: compose $D \circ G$ (feed generator output state into discriminator unitaries, then measure).
- **Score:** map expectation to probability, e.g., $p_{\text{real}}=\frac{1+\langle Z_0\rangle}{2}$ or a small readout head measuring multiple Pauli terms.

> Using a **classical discriminator** is the most practical: take bitstrings from the quantum generator and train a small MLP. It stabilizes training and avoids deep $D(\phi)$.

---

## 4) Training objectives & optimizers

### 4.1 Standard GAN (BCE)
- Discriminator loss  
  $$
  \mathcal{L}_D = -\mathbb{E}_{x\sim p_{\text{data}}}\log D(x) \;-\; \mathbb{E}_{\tilde{x}\sim G_\theta}\log\big(1-D(\tilde{x})\big).
  $$
- Generator loss (non-saturating)  
  $$
  \mathcal{L}_G = -\mathbb{E}_{\tilde{x}\sim G_\theta}\log D(\tilde{x}).
  $$

### 4.2 Wasserstein GAN (often more stable)
- Discriminator becomes a **critic** $f_\phi$ with 1-Lipschitz constraint; loss  
  $$
  \mathcal{L}_D = \mathbb{E}_{\tilde{x}} f_\phi(\tilde{x}) - \mathbb{E}_{x} f_\phi(x) + \lambda\,\text{GP},
  $$
  where **GP** is gradient penalty (classical discriminator) or spectral norm proxy.
- Generator loss $\mathcal{L}_G = -\mathbb{E}_{\tilde{x}} f_\phi(\tilde{x})$.

### 4.3 Optimizers & gradients
- **Parameter-shift** for EstimatorQNN expectations; **SPSA** is robust with few shots.  
- **Learning rates**: start small (1e-2 → 1e-3).  
- **Shot budgeting:** 256–1024 early; increase near convergence.

---

## 5) Stabilization: mode collapse & barren plateaus
**Mode collapse (GAN-typical)**  
- Use **label smoothing**, **instance noise** (Gaussian noise on logits), **minibatch discrimination**, or **Wasserstein** losses.

**Barren plateaus (quantum-typical)**  
- **Layer-wise growth** (grow depth gradually).  
- **Problem-aware ansatz** (e.g., limited entanglers).  
- **Identity initialization** (start near identity).  
- **Shallow circuits + SPSA**.

---

## 6) Small architectures you can train today

### 6.1 1D mixture (2-qubit generator)
- Target: discrete 4-bin distribution approximating a 1D Gaussian mixture.  
- $G(\theta)$: 2 qubits, `ry/rz` + `cx` (1 rep).  
- $D$: classical MLP(4→32→1) on one-hot bitstrings.

### 6.2 Bars-and-Stripes BAS(2×2) (4 qubits)
- Target bitstrings: `0000, 1111, 0011, 1100, 0101, 1010`.  
- Generator: 4-qubit TwoLocal with ring entanglement.  
- Discriminator: classical MLP (6 real examples + generator samples per batch).

> Keep **depth ≤ 2 reps**, shots 1024, and run on **Aer** with optional noise.

---

## 7) Qiskit implementation (modern primitives)

> We use **primitives** (`Sampler`, `Estimator`) + **TorchConnector**. For stability, this example uses a **quantum generator** + **classical discriminator** (QG-CD), which is the easiest to get working and debug.

### 7.1 Quantum generator as a PyTorch module
```python
# pip install qiskit qiskit-aer qiskit-machine-learning torch
import torch, numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit.library import TwoLocal
from qiskit_aer.primitives import Sampler
from qiskit_machine_learning.neural_networks import SamplerQNN
from qiskit_machine_learning.connectors import TorchConnector

n_qubits = 4
ansatz = TwoLocal(n_qubits,
                  rotation_blocks=['ry','rz'],
                  entanglement_blocks='cx',
                  entanglement='linear',
                  reps=1, insert_barriers=False)

# Generator QNN: outputs probabilities over {0,1}^n via the Sampler
g_qnn = SamplerQNN(
    circuit=ansatz,
    input_params=[],                    # no classical inputs, pure generative
    weight_params=list(ansatz.parameters),
    sparse=False,                       # dense probability vector
    interpret=None                      # raw bitstrings
)
G = TorchConnector(g_qnn)               # torch.nn.Module; params are quantum weights

# Helper: sample bitstrings from the generator
def sample_from_G(Gmodule, num_samples=256, device='cpu'):
    with torch.no_grad():
        probs = Gmodule()               # shape [2**n]
        probs = probs / probs.sum()     # normalize (guard against numerical drift)
        # Multinomial sampling on CPU/GPU
        idx = torch.multinomial(probs, num_samples=num_samples, replacement=True)
        # Convert indices to bitstrings
        bitstrings = [format(i.item(), f'0{n_qubits}b') for i in idx]
    return bitstrings
```

### 7.2 Classical discriminator & GAN losses
```python
import torch.nn as nn
class D_MLP(nn.Module):
    def __init__(self, n_bits):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_bits, 32), nn.LeakyReLU(0.2),
            nn.Linear(32, 16), nn.LeakyReLU(0.2),
            nn.Linear(16, 1)
        )
    def forward(self, x_bits01):  # x in {0,1}^n as float
        return self.net(x_bits01).squeeze(-1)  # logits

D = D_MLP(n_qubits)

bce = nn.BCEWithLogitsLoss()

def bits_to_tensor(bitstrings, device='cpu'):
    X = torch.tensor([[int(b) for b in s] for s in bitstrings], dtype=torch.float32, device=device)
    return X

# Real dataset builder (e.g., BAS(2x2))
BAS = ['0000','1111','0011','1100','0101','1010']
def real_batch(batch_size=256):
    idx = np.random.randint(0, len(BAS), size=batch_size)
    return [BAS[i] for i in idx]
```

### 7.3 Training loop (QG-CD, BCE)
```python
g_opt = torch.optim.Adam(G.parameters(), lr=1e-2)
d_opt = torch.optim.Adam(D.parameters(), lr=1e-3)

for step in range(2000):
    # ---- Train D ----
    D.train(); G.eval()
    xb = bits_to_tensor(real_batch(256))     # real
    zb = bits_to_tensor(sample_from_G(G, 256))  # fake from quantum generator

    logits_r = D(xb); logits_f = D(zb)
    loss_d = bce(logits_r, torch.ones_like(logits_r)*0.9) + \
             bce(logits_f, torch.zeros_like(logits_f))
    d_opt.zero_grad(); loss_d.backward(); d_opt.step()

    # ---- Train G ----
    D.eval(); G.train()
    zb = bits_to_tensor(sample_from_G(G, 256))
    logits_f = D(zb)
    # non-saturating generator loss: maximize D(fake)
    loss_g = bce(logits_f, torch.ones_like(logits_f))
    g_opt.zero_grad(); loss_g.backward(); g_opt.step()

    if step % 200 == 0:
        with torch.no_grad():
            probs = G().cpu().numpy()
            kl = 0.0
            # crude monitoring vs uniform over BAS modes
            p_target = np.zeros_like(probs)
            for s in BAS:
                p_target[int(s, 2)] = 1.0/len(BAS)
            eps = 1e-9
            kl = np.sum(p_target * (np.log(p_target+eps) - np.log(probs+eps)))
        print(f"step {step:4d} | L_D {loss_d.item():.3f}  L_G {loss_g.item():.3f}  KL≈{kl:.2f}")
```

> **Notes.**  
> • `SamplerQNN` returns a probability vector; we sample **classically** to build fake batches.  
> • For **Wasserstein**: replace `D` head and BCE with critic loss + gradient penalty (classical).  
> • For a **quantum discriminator**, build a PQC `D(φ)` and two Estimator problems: `D∘Prep(x)` (real) and `D∘G(θ)` (fake). Use two `EstimatorQNN`s (or `Estimator` directly) and optimize $\theta,\phi$ jointly.

---

## 8) Building QGAN architectures
- **Generator**: 4–6 qubits, `TwoLocal(ry,rz,cx)`, 1–2 reps; optional data re-upload with classical noise $z$.  
- **Discriminator**:  
  - **Classical** MLP (recommended on NISQ), or  
  - **Quantum** PQC with an expectation head; keep depth small, share parameters across qubits if needed.  
- **Encoding real data**:  
  - **Bitstrings**: basis state prep is trivial.  
  - **Continuous 1D/2D**: discretize into bins → bitstrings, or use amplitude/angle encoding + a classical post-processing head.

---

## 9) Assessing generated samples

### For classical bitstrings
- **Histogram distance**: KL divergence, Jensen–Shannon (JS), **Earth Mover’s Distance** (EMD).  
- **Mode coverage**: fraction of target modes (e.g., BAS patterns) with probability above a threshold.  
- **Fréchet-like** scores need embeddings (for images).

### For quantum states
- **State fidelity** vs target (simulator) or **MMD** with quantum kernel.

**Reporting.** Always include **shots**, **circuit depth**, and **wall-clock** for sampling.

---

## 10) Practical tips & pitfalls
- **Keep it shallow.** 1–2 reps; increase only if loss plateaus.  
- **Readout mitigation** improves discriminator signals when using quantum D.  
- **Batching & caching.** Cache circuit transpiles; batch `Sampler` calls.  
- **Label smoothing** (0.9 for real) stabilizes BCE.  
- **Learning-rate schedule**: decay by 0.5 every few hundred steps.  
- **Entropy floor.** Encourage generator entropy early (e.g., add $-\lambda H(p_\theta)$ regularizer) to avoid collapse.  
- **Fair baselines.** Compare to classical GAN/VAEs with matched parameter count.

---

## 11) Mini-exercises (answers in Appendix)
1. **BAS likelihoods.** For BAS(2×2), list the 6 valid bitstrings and write a projector-based observable whose expectation equals the model’s total BAS probability mass.  
2. **Mode collapse detector.** Propose a scalar statistic computed from generator histograms that signals collapse early in training.  
3. **Parameter-shift.** Show that for an expectation $\langle O\rangle$, $\partial_\theta \langle O\rangle = \tfrac12 \big(\langle O\rangle_{\theta+\pi/2} - \langle O\rangle_{\theta-\pi/2}\big)$ for Pauli rotation gates.  
4. **Shot budgeting.** If you want std. error ≤ 0.02 on a Bernoulli head (p≈0.7), estimate the shots needed per estimate.  
5. **WGAN-GP (classical D).** Write the gradient penalty term and explain how to approximate it when fake samples come as bitstrings.

---

## 12) Summary (Session 21)
- QGANs pair a **quantum generator** (and optionally quantum discriminator) with a **variational** min–max objective.  
- On NISQ, the **QG-CD** pattern is the most practical: quantum sampling + classical scoring.  
- Stabilize with **shallow circuits**, **label smoothing**, **Wasserstein losses**, and **SPSA/Adam**.  
- Evaluate with **histogram distances**, **mode coverage**, and report **shots/depth**.  
- Good first targets: **1D mixtures** and **BAS(2×2/3×3)**.

---

## 13) Looking ahead
- **Next Session:** Quantum Autoencoders — compressing quantum/classical distributions with PQCs and reconstruction losses.  
- **Homework 5 (QGAN):**  
  1) Train a 4-qubit generator on BAS(2×2) with BCE; report JS divergence vs steps and show mode coverage.  
  2) Swap the MLP discriminator for a 2-layer PQC and compare stability and sample quality (same shot budget).  
  3) (Bonus) Try WGAN with gradient penalty (classical D) and compare to BCE.

---

## Appendix — solutions (sketch)

1. **BAS projector.** Let $\mathcal{S}$ be the 6 BAS bitstrings. Define $P_{\text{BAS}}=\sum_{s\in\mathcal{S}} |s\rangle\langle s|$. Then $\langle P_{\text{BAS}}\rangle$ equals the generator’s total probability mass on BAS. Implement with a sum of Pauli strings or estimate by summing measured frequencies over $\mathcal{S}$.  
2. **Collapse statistic.** Entropy $H(p_\theta)=-\sum_x p_\theta(x)\log p_\theta(x)$. A sharp drop early (with constant loss) flags collapse; also monitor **effective support size** $\exp(H)$.  
3. **Param-shift.** For $U(\theta)=e^{-i\theta P/2}$ with $P^2=I$, $f(\theta)=\langle 0|U^\dagger O U|0\rangle$. Using eigenspectrum $\{\pm1\}$ of $P$, the derivative equals half-difference at shifts $\pm \pi/2$.  
4. **Shots.** Std. error $\sqrt{p(1-p)/S}\le 0.02\Rightarrow S\ge p(1-p)/0.0004\approx 0.21/0.0004\approx 525$. Use **≥ 512–1024** shots per estimate.  
5. **WGAN-GP.** $\text{GP}=\mathbb{E}_{\hat{x}}\big(\|\nabla_{\hat{x}} f_\phi(\hat{x})\|_2 - 1\big)^2$, with $\hat{x}=\epsilon x + (1-\epsilon)\tilde{x}$. For bitstrings, relax to $[0,1]^n$ by adding small Gaussian noise or use **spectral normalization** on the MLP as a proxy.
````
